In [1]:
# !pip install laspy[lazrs]
# !pip install pdal

In [2]:
from __future__ import annotations
import time
import argparse
import sys
from pathlib import Path

import laspy
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from scipy import ndimage as ndi
from scipy.ndimage import uniform_filter

# sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import os
import matplotlib
import math
import numpy as np
from pathlib import Path
from glob import glob
import laspy
import sys
import whitebox
from affine import Affine        # or: from rasterio import Affine
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))
print(parent_dir)
from _common import DERIV, DST_CRS, ROOT, make_profile, read_tif, run_pdal, write_tif, path_for

C:\Users\colto\Documents\GitHub\lidar_project\notebooks\wellsight_v2


In [3]:
NEW_TILES_NAME = "OTHER_DATA"
LAZ_FOLDER = "test_section"
LIDAR_TYPE = ".laz"
# LAZ_PATH = Path.cwd().parent.parent.parent/"data/_source/lidar/westernpa/separate_sections"/LAZ_FOLDER
LAZ_PATH = Path.cwd().parent.parent.parent/"data/_source/lidar/westernpa/OTHER_DATA"
OUTPUT_PATH = Path.cwd().parent.parent.parent/"data"/NEW_TILES_NAME /"derived"
TEAL = np.array([0, 158, 162], float)     # concave  (valley / pit)  -> depression tint
GRAY = np.array([138, 138, 138], float)   # flat
YELLOW = np.array([254, 255, 172], float) # convex   (ridge)         -> ridge tint
WHITE = np.array([255, 255, 255], float)  # slope 0  (flat)
RED = np.array([182, 39, 0], float)       # slope high (steep)       -> vivid red
def out(stem:str,sfx: str, ext:str = "tif") -> Path:
    return out_dir / f"{stem}_{sfx}.{ext}"

def disk_kernel(r_cells: float) -> np.ndarray:
    r = int(round(r_cells))
    y, x = np.ogrid[-r:r+1, -r:r+1]
    return (x * x + y * y) <= r * r
def _nanmax_disk(a: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    big = np.where(np.isfinite(a), a, -np.inf)
    r = ndi.maximum_filter(big, footprint=kernel, mode="nearest")
    return np.where(np.isfinite(r), r, np.nan).astype(np.float32)
def _nanmin_disk(a: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    small = np.where(np.isfinite(a), a, np.inf)
    r = ndi.minimum_filter(small, footprint=kernel, mode="nearest")
    return np.where(np.isfinite(r), r, np.nan).astype(np.float32)
def _read_data(path, out_shape = None):
    with rasterio.open(path) as ds:
        a = ds.read(1,out_shape = out_shape, resampling = Resampling.bilinear).astype("float32")
        a = np.where(a == ds.nodata, np.nan, a)
        return a, ds.profile

Stitch together the laz if there are more than 1

In [4]:
def resolve_tiles(spec: str) -> list[Path]:
    """Resolve a glob (relative to repo root) OR comma-separated paths."""
    if "," in spec:
        return [(Path(p.strip()) if Path(p.strip()).is_absolute() else ROOT / p.strip())
                for p in spec.split(",")]
    p = Path(spec)
    if p.is_absolute():
        return sorted(Path(p.anchor).glob(str(p.relative_to(p.anchor)).replace("\\", "/")))
    return sorted(ROOT.glob(spec))

def stitcher(
    tiles: list[Path],
    *,
    x0: float, y0: float, x1: float, y1: float,
    res: float, sfx: str, dst_crs: str, width: int, height: int,
    src_crs: str | None = None,
    merge_path: Path | None = None,
    skip_existing: bool = True,
    out_dir: Path | None = None) -> Path:

    # # width = int(round((x1-x0)/res))
    # # height = int(round((y1-y0)/res))
    # transform = from_origin(x0,y1,res,res)


    print(f"\n=== {sfx} ===  grid {width}x{height} @ {res} m  CRS={dst_crs}  tiles={len(tiles)}")
    print(f"  output dir: {out_dir.relative_to(ROOT)}")

    if len(tiles) == 1 and src_crs is None:
        las_path = tiles[0]
    else:
        if merge_path is None:
            merge_path = LAZ_PATH / f"_merged_{sfx}.las"
        merge_path.parent.mkdir(parents=True, exist_ok=True)
        if merge_path.exists() and skip_existing:
            print(f"  merge: reusing existing {merge_path.name}")
        else:
            stages: list = [{"type": "readers.las", "filename": str(p)} for p in tiles]
            if src_crs:
                stages.append({"type": "filters.reprojection", "in_srs": src_crs, "out_srs": dst_crs})
            stages.append({"type": "filters.crop", "bounds": f"([{x0}, {x1}], [{y0}, {y1}])"})
            stages.append({"type": "writers.las", "filename": str(merge_path),
                           "minor_version": 4, "dataformat_id": 7, "a_srs": dst_crs,
                           "compression": "false",
                           "offset_x": "auto", "offset_y": "auto", "offset_z": "auto",
                           "scale_x": 0.01, "scale_y": 0.01, "scale_z": 0.01})
            run_pdal(stages, label = f"merge_{sfx}")
            print(f"  merged: {merge_path.stat().st_size/1e9:.2f} GB")
        las_path = merge_path
    return las_path





DEM

In [5]:
def DEM_maker(
    *,
    x0: float, y0: float,
        W: float, H: float,
    res: float, sfx: str, las_path: Path | None = None,
    skip_existing: bool = True,
    dem_method: str = "delaunay",
    ) -> np.ndarray:
    dem_path = out("dem", sfx = sfx)

    if not (dem_path.exists() and skip_existing):
        ground = [{'type': "readers.las", "filename": str(las_path)},
                  {'type': "filters.range", "limits": "Classification[2:2]"}]
        if dem_method == "gdal":
            stages = ground + [{'type': 'writers.gdal', 'filename': str(dem_path),
                                'output_type': 'idw', 'resolution':res,
                                'origin_x':x0, 'origin_y': y0, 'width': W, 'height': H,
                                'window_size': 3, 'data_type': 'float32'}]
        else:
            stages = ground + [
                {"type": "filters.delaunay"},
                {"type": "filters.faceraster",
                 "resolution": res, "origin_x": x0, "origin_y": y0,
                 "width": W, "height": H},
                {"type": "writers.raster", "filename": str(dem_path),
                 "data_type": "float32"},
            ]
        run_pdal(stages, label = f"dem_{sfx}")
        dem = read_tif(dem_path)
        print(f"  DEM: nan={100*np.isnan(dem).mean():.2f}%  "
        f"z={np.nanmin(dem):.1f}..{np.nanmax(dem):.1f} m")
    return dem


Openess Only

In [6]:
def openness(z: np.ndarray, *, L_cells: int, cellsize: float):
    dirs = [(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1),(-1,-1)]
    valid = np.isfinite(z)
    z0 = np.where(valid, z, 0).astype(np.float32)
    phi = np.zeros_like(z, dtype=np.float32)
    psi = np.zeros_like(z, dtype=np.float32)
    for dr, dc in dirs:
        step = cellsize * np.hypot(dr, dc)
        mtu = np.full_like(z, -np.inf, dtype=np.float32)
        mtd = np.full_like(z, np.inf, dtype=np.float32)
        for k in range(1, L_cells + 1):
            zs = np.roll(z0, shift=(dr * k, dc * k), axis=(0, 1))
            vs = np.roll(valid, shift=(dr * k, dc * k), axis=(0, 1))
            if dr > 0:   vs[:dr*k, :] = False
            elif dr < 0: vs[dr*k:, :] = False
            if dc > 0:   vs[:, :dc*k] = False
            elif dc < 0: vs[:, dc*k:] = False
            ta = np.where(vs, (zs - z0) / (k * step), np.nan).astype(np.float32)
            np.fmax(mtu, ta, out=mtu, where=vs)
            np.fmin(mtd, ta, out=mtd, where=vs)
        phi += (np.pi / 2 - np.arctan(np.where(np.isfinite(mtu), mtu, 0))).astype(np.float32)
        psi += (np.pi / 2 + np.arctan(np.where(np.isfinite(mtd), mtd, 0))).astype(np.float32)
    phi = np.degrees(phi / 8).astype(np.float32)
    psi = np.degrees(psi / 8).astype(np.float32)
    phi[~valid] = np.nan; psi[~valid] = np.nan
    return phi, psi
def openness_process(
    *,
    res: float, sfx: str, dst_crs: str, dem:np.ndarray, transform: Affine) -> np.ndarray:
    op_pos, op_neg = openness(dem, L_cells=int(25/res), cellsize = res)
    write_tif(out("openness_pos", sfx = sfx), op_pos, transform = transform, crs=dst_crs)
    write_tif(out("openness_neg", sfx = sfx), op_neg, transform = transform, crs=dst_crs)
    print(f"  openness_only: wrote openness_pos + openness_neg for {sfx}")
    return op_pos, op_neg

DSM + CHM

In [7]:
def dsm_chm_process(sfx: str, res: float, x0: float, y0: float,
        W: float, H: float, dst_crs: str, skip_existing: bool = True):
    dsm_path = out("dsm", sfx = sfx)
    if not (dsm_path.exists() and skip_existing):
        run_pdal([
            {"type": "readers.las", "filename": str(las_path)},
            {"type": "filters.range", "limits": "ReturnNumber[1:1]"},
            {"type": "writers.gdal", "filename": str(dsm_path),
             "output_type": "max", "resolution": res,
             "origin_x": x0, "origin_y": y0, "width": W, "height": H,
             "data_type": "float32"},
        ], label=f"dsm_{sfx}")
    dsm = read_tif(dsm_path)
    chm = np.where(np.isnan(dsm) | np.isnan(dem), np.nan,
                   np.maximum(dsm - dem, 0)).astype(np.float32)
    write_tif(out("chm", sfx = sfx), chm, transform=transform, crs=dst_crs)
    return dsm, chm

Density Intensity

In [8]:
def density_intensity(sfx: str, res: float, x0: float, y1: float,
        W: float, H: float, dst_crs: str):
    density_flat = np.zeros(H*W)
    sum_i = np.zeros(H*W, dtype=np.float64)
    count = np.zeros(H*W, dtype=np.float64)
    with laspy.open(str(las_path)) as lf:
        for pts in lf.chunk_iterator(5_000_000):
            cls=np.asarray(pts.classification)
            gm = cls == 2
            if not gm.any():
                continue
            xs = np.asarray(pts.x)[gm]; ys = np.asarray(pts.y)[gm]
            iv = np.asarray(pts.intensity).astype(np.float64)[gm]
            col = np.floor((xs-x0)/res).astype(np.int64)
            row = np.floor((y1-ys)/res).astype(np.int64)
            ok = (col >= 0) & (col < W) & (row >= 0) & (row <H)
            fi = row[ok] * W + col[ok]
            density_flat += np.bincount(fi, minlength = H * W)
            sum_i += np.bincount(fi, weights=iv[ok], minlength = H * W)
            count += np.bincount(fi, minlength = H*W)
    density_map = density_flat.reshape(H,W).astype(np.uint16)
    write_tif(out("ground_density", sfx = sfx), density_map, transform = transform, crs = dst_crs, dtype = "uint16", nodata = 0)
    mean_i = np.full(H*W, np.nan, dtype = np.float32)
    with np.errstate(invalid = "ignore"):
        np.divide(sum_i, count, out=mean_i, where=count > 0)
    intensity_map = mean_i.reshape(H,W)
    write_tif(out("intensity_ground", sfx = sfx), intensity_map, transform = transform, crs = dst_crs)
    del density_flat, sum_i,count, mean_i
    return density_map, intensity_map

WBT Hillshade and Slope


In [9]:
def hillshade_slope_process(sfx: str, out_dir: Path):
    wbt = whitebox.WhiteboxTools()
    wbt.set_working_dir(str(out_dir.resolve()))
    wbt.set_verbose_mode(False)
    hs_name, sl_name = f"hillshade_{sfx}.tif", f"slope_{sfx}.tif"
    wbt.hillshade(dem=f"dem_{sfx}.tif", output=hs_name, azimuth=315.0, altitude=45)
    wbt.slope(dem=f"dem_{sfx}.tif", output = sl_name, units="degrees")
    print(f"  wrote hillshade_{sfx} + slope_{sfx}")
    slope_file, prof = _read_data(out_dir / sl_name)
    HS_file, prof = _read_data(out_dir / hs_name)
    return slope_file, HS_file, prof


Roughness

In [10]:
def nanmean_filter(a: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    valid = np.isfinite(a).astype(np.float32)
    a0 = np.where(valid.astype(bool), a, 0).astype(np.float32)
    k = kernel.astype(np.float32)
    s = ndi.convolve(a0, k, mode="nearest")
    c = ndi.convolve(valid, k, mode="nearest")
    out = np.full_like(a, np.nan, dtype=np.float32)
    np.divide(s, c, out=out, where=c > 0)
    return out
def derivatives_process(sfx: str, res: float, dst_crs: str):
    def tpi(z:np.ndarray, r_m: float) -> np.ndarray:
        return (z-nanmean_filter(z, disk_kernel(r_m / res))).astype(np.float32)
    lrm_dict = {}
    WIN = 5
    k = np.ones((WIN, WIN), dtype=np.float32)
    v = np.isfinite(dem).astype(np.float32)
    z0 = np.where(v.astype(bool), dem, 0).astype(np.float32)
    s = ndi.convolve(z0, k, mode = "nearest")
    s2 = ndi.convolve(z0 * z0, k, mode="nearest")
    n = ndi.convolve(v, k, mode = "nearest")
    var = np.where(n > 1, (s2 - s * s / np.maximum(n,1))/ np.maximum(n-1, 1), np.nan)
    rough = np.sqrt(np.clip(var, 0, None)).astype(np.float32)
    rough[n < WIN * WIN] = np.nan
    t05 = tpi(dem, 5.0)
    t15 = tpi(dem, 15.0)
    t25 = tpi(dem, 25.0)
    gy, gx = np.gradient(t15, res)
    tpi_grad_mag = np.hypot(gx, gy).astype(np.float32)
    tpi_grad_dir = (np.degrees(np.arctan2(gx, -gy)) % 360).astype(np.float32)
    rk = disk_kernel(10 / res)
    lr = (_nanmax_disk(dem, rk) - _nanmin_disk(dem, rk)).astype(np.float32)
    for size in (3,5,11,25):
        valid = np.isfinite(dem).astype(np.float32)
        zo = np.where(valid.astype(bool), dem, 0).astype(np.float32)
        sm = uniform_filter(zo, size=size, mode = "nearest")
        sc = uniform_filter(valid, size=size, mode = "nearest")
        smooth = np.where(sc > 0, sm/sc, np.nan)
        lrm = (dem - smooth).astype(np.float32)
        write_tif(out(f"lrm_{size}", sfx=sfx), lrm, transform = transform, crs = dst_crs)
        lrm_dict[f"lrm_{size}"] = lrm
    write_tif(out("roughness_5", sfx=sfx), rough, transform = transform, crs = dst_crs)
    write_tif(out("local_relief_10", sfx=sfx), lr, transform = transform, crs = dst_crs)
    write_tif(out("tpi_05", sfx=sfx), t05, transform=transform, crs=dst_crs)
    write_tif(out("tpi_15", sfx=sfx), t15, transform=transform, crs=dst_crs)
    write_tif(out("tpi_25", sfx=sfx), t25, transform=transform, crs=dst_crs)
    write_tif(out("tpi_grad_mag", sfx=sfx), tpi_grad_mag, transform=transform, crs=dst_crs)
    write_tif(out("tpi_grad_dir", sfx=sfx), tpi_grad_dir, transform=transform, crs=dst_crs)
    return lrm_dict, rough, lr,t05, t15, t25, tpi_grad_mag, tpi_grad_dir


RED RELIEF IMAGE MAP

In [11]:
def rrim_process(sfx: str, dst_crs: str, slope_file: np.ndarray, open_pos: np.ndarray, open_neg: np.ndarray, ):
    def _diverge(t, neg_color, pos_color, mid_color):
        """t in [-1,1] -> RGB via mid->neg (t<0) / mid->pos (t>0). Returns (...,3)."""
        t = t[..., None]
        lo = mid_color + (neg_color - mid_color) * (-t)   # t<0
        hi = mid_color + (pos_color - mid_color) * (t)    # t>=0
        return np.where(t < 0, lo, hi)
    base_brightness=18.0
    slope_hi=40.0
    do_pct=98.0
        # --- Simple Red Relief color stops (Auld-Thomas 2022), reused as the RRIM palette ---
    TEAL = np.array([0, 158, 162], float)     # concave  (valley / pit)  -> depression tint
    GRAY = np.array([138, 138, 138], float)   # flat
    YELLOW = np.array([254, 255, 172], float) # convex   (ridge)         -> ridge tint
    WHITE = np.array([255, 255, 255], float)  # slope 0  (flat)
    RED = np.array([182, 39, 0], float)       # slope high (steep)       -> vivid red
    valid = np.isfinite(slope_file)
    do = (open_pos - open_neg) / 2.0
    valid &= np.isfinite(open_pos) & np.isfinite(open_neg)
    dlim = np.nanpercentile(np.abs(do[np.isfinite(do)]), do_pct)

    t = np.clip(do/dlim, -1, 1)
    t = np.nan_to_num(t, nan=0)
    base = _diverge(t, TEAL, YELLOW, GRAY) + base_brightness
    base = np.clip(base, 0, 255)


    u = np.clip(np.nan_to_num(slope_file, nan=0.0) / slope_hi, 0, 1)[..., None]
    slope_rgb = WHITE + (RED-WHITE) * u
    rrim = base * slope_rgb / 255
    rrim = np.clip(rrim, 0, 255)
    rrim[~valid] = 255
    rrim = rrim.astype("uint8")
    print(f"  rrim: base=differential openness (+/-{dlim:.2f} deg)  slope_hi={slope_hi}")
    print(rrim.ndim)
    write_tif(out("rrim_openness", sfx=sfx), arr = rrim, transform=transform, crs=dst_crs, rgb_bool = True)
    return rrim

In [12]:
FILE_USED = [i for i in LAZ_PATH.iterdir()
                if i.is_file() and i.suffix.lower() in LIDAR_TYPE]
min_x, min_y = float("inf"), float("inf")
max_x, max_y = float("-inf"), float("-inf")
for filepath in FILE_USED:
    with laspy.open(filepath) as f:
        hdr = f.header
        min_x = min(min_x, hdr.min[0])
        min_y = min(min_y, hdr.min[1])
        max_x = max(max_x, hdr.max[0])
        max_y = max(max_y, hdr.max[1])
BBOX_COORDS = f"{min_x:.2f},{min_y:.2f},{max_x:.2f},{max_y:.2f}"

# def main() -> int:
ap = argparse.ArgumentParser()
ap.add_argument("--tiles", required=True, help="Glob or comma-separated paths")
ap.add_argument("--bbox", required=True, help="X0,Y0,X1,Y1 in target CRS (m)")
ap.add_argument("--suffix", required=True, help="Output suffix (e.g. 9t_1m)")
ap.add_argument("--res", type=float, default=1.0)
ap.add_argument("--crs", default=DST_CRS, help=f"Target CRS (default {DST_CRS})")
ap.add_argument("--src-crs", default=None)
ap.add_argument("--merge-path", default=None)
ap.add_argument("--overwrite", action="store_true")
ap.add_argument("--out-dir", default=None)
ap.add_argument("--dem-method", default="delaunay", choices=["delaunay", "gdal"])

# Pass pre-defined variables as strings
args = ap.parse_args([
    "--tiles", str(LAZ_PATH / f"*{LIDAR_TYPE}"),     # E.g., "/path/to/data/_source/lidar/westernpa/*.las"
    "--bbox", BBOX_COORDS,                          # E.g., "500000,4000000,505000,4005000"
    "--suffix", NEW_TILES_NAME,                     # E.g., "FOO"
    "--out-dir", str(OUTPUT_PATH),                  # E.g., "/path/to/data/FOO"
    "--overwrite"                                   # Optional flag
])
out_dir = Path(args.out_dir) if args.out_dir else None
# print(out_dir)
# if out_dir is None:
#     out_dir = OUTPUT_PATH
#     # print(out_dir)
out_dir = Path(args.out_dir) if args.out_dir else OUTPUT_PATH
out_dir.mkdir(parents=True, exist_ok=True)
# print(out_dir)
# print(out_dir)
tiles = resolve_tiles(args.tiles)
x0, y0, x1, y1 = (float(v) for v in args.bbox.split(","))
width = int(round((x1-x0)/args.res))
height = int(round((y1-y0)/args.res))
transform = from_origin(x0,y1,args.res,args.res)
merge = Path(args.merge_path) if args.merge_path else None




In [13]:
print(tiles)
las_path = stitcher(
    tiles, x0=x0, y0=y0, x1=x1, y1=y1,width= width, height= height,
    res=args.res, sfx=args.suffix, dst_crs=args.crs,
    src_crs=args.src_crs, merge_path=merge,
    skip_existing=(not args.overwrite),
    out_dir=out_dir)

[WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/_source/lidar/westernpa/OTHER_DATA/USGS_LPC_PA_WesternPA_2019_D20_17TPF619593.laz'), WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/_source/lidar/westernpa/OTHER_DATA/USGS_LPC_PA_WesternPA_2019_D20_17TPF619594.laz'), WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/_source/lidar/westernpa/OTHER_DATA/USGS_LPC_PA_WesternPA_2019_D20_17TPF619596.laz'), WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/_source/lidar/westernpa/OTHER_DATA/USGS_LPC_PA_WesternPA_2019_D20_17TPF621593.laz'), WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/_source/lidar/westernpa/OTHER_DATA/USGS_LPC_PA_WesternPA_2019_D20_17TPF621594.laz'), WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/_source/lidar/westernpa/OTHER_DATA/USGS_LPC_PA_WesternPA_2019_D20_17TPF621596.laz'), WindowsPath('C:/Users/colto/Documents/GitHub/lidar_project/data/_source/lidar/westernpa/OTHER_DATA/USGS_L

In [14]:
dem = DEM_maker(x0=x0, y0=y0,
                W=width, H=height,
                res=args.res, sfx=args.suffix,
                las_path = las_path, skip_existing=(not args.overwrite),
                dem_method=args.dem_method)

  [dem_OTHER_DATA] rc=0 in 379.1s
  DEM: nan=0.00%  z=328.8..509.5 m


In [15]:
open_pos, open_neg = openness_process(res=args.res, sfx=args.suffix,dst_crs=args.crs, dem = dem, transform = transform)

  openness_only: wrote openness_pos + openness_neg for OTHER_DATA


In [16]:
dsm, chm = dsm_chm_process(res=args.res, sfx=args.suffix, x0=x0, y0=y0,
                W=width, H=height, dst_crs=args.crs, skip_existing=(not args.overwrite))

  [dsm_OTHER_DATA] rc=0 in 126.7s


Density and Intensity

In [17]:
density, intensity_map = density_intensity(res=args.res, sfx=args.suffix, x0=x0, y1=y1,
                W=width, H=height, dst_crs=args.crs)



In [18]:
slope_file, HS_file, prof = hillshade_slope_process(sfx=args.suffix, out_dir=out_dir)

  wrote hillshade_OTHER_DATA + slope_OTHER_DATA


In [19]:
lrm, rough, lr,t05, t15, t25, tpi_grad_mag, tpi_grad_dir = derivatives_process(res=args.res, sfx=args.suffix, dst_crs=args.crs)

RED RELIEF IMAGE MAP

In [20]:
rrim = rrim_process(sfx=args.suffix, dst_crs=args.crs, slope_file = slope_file, open_pos = open_pos, open_neg = open_neg)

  rrim: base=differential openness (+/-4.72 deg)  slope_hi=40.0
3
